## Limpieza de datos, archivo: Orders items

In [2]:
import pandas as pd
import numpy as np

RAW = "../../data/raw/"

order_items = pd.read_csv(
    RAW + "olist_order_items_dataset.csv",
    parse_dates=["shipping_limit_date"]  # fecha límite de envío
)

In [6]:
# ===================================
# Duplicados (seguridad)
# ===================================
order_items = order_items.drop_duplicates(subset=["order_id", "order_item_id"]).copy()
print(f"Tras drop_duplicates: {len(order_items):,}")

datetime_cols = ["shipping_limit_date"]
numeric_cols  = ["price", "freight_value"]  # métricas de interés
id_like_cols  = ["order_item_id"]   

Tras drop_duplicates: 112,650


In [7]:
# ===================================
# Outliers ±3σ (solo en métricas: price, freight_value)
# ===================================
def remove_outliers_3sigma(df, cols):
    if not cols:
        return df
    mask = pd.Series(True, index=df.index)
    for c in cols:
        x = df[c].astype(float)
        mu = x.mean()
        sigma = x.std(ddof=0)
        # si sigma=0 o NaN, no filtramos esa columna
        if np.isnan(mu) or np.isnan(sigma) or sigma == 0:
            continue
        mask &= (x >= mu - 3*sigma) & (x <= mu + 3*sigma)
    return df[mask].copy()

before = len(order_items)
order_items_clean = remove_outliers_3sigma(order_items, numeric_cols)
print(f"Eliminadas por outliers (±3σ) en {numeric_cols}: {before - len(order_items_clean):,}")

Eliminadas por outliers (±3σ) en ['price', 'freight_value']: 3,558


In [8]:
# ===================================
# 6) Resultado
# ===================================
print("\nResumen final order_items_clean")
print(f"Filas finales: {len(order_items_clean):,}")
print(order_items_clean.dtypes)


Resumen final order_items_clean
Filas finales: 109,092
order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object


In [10]:
# Vista rápida
order_items_clean.head(100)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
97,0035e6b7ade84b3f5b86bd49814793df,1,71a5f1c2a5fd9889ef26b5ac22aec9c6,537eb890efff034a88679788b647c564,2018-02-27 03:31:08,19.90,14.10
98,0036757472ece3dde52fd4bfd929c90e,1,4c1bbc12438daec98a77243c2bf7a3ba,7c67e1448b00f6e969d365cea6b010ab,2018-08-08 15:10:11,136.99,66.04
99,0036887767dea4bd43b1a88cd0d9477a,1,3a264b078bf20e98f315ff65c23fa263,46dc3b2cc0980fb8ec44634e21d2718e,2017-10-19 01:07:30,399.99,23.64
100,00378c6c981f234634c0b9d6128df6dd,1,38fa750a3a3b3204f169c86a3284d387,218d46b86c1881d022bce9c68a7d4b15,2018-02-08 19:50:27,41.00,11.85
